# 第21章　可視化の実装 ― オーバーレイと結果提示

**『医療診断支援AI開発　基礎編 ― 自分で作る（基礎編）』のコード**

本文に載っているコードを、章の順にそのまま収めています。紙面のコードは読んで理解するためのもの、こちらは動かすためのものです。

- Python 以外（シェル・YAML・Dockerfile など）は、実行環境が違うので**コードセルにせず、そのまま読める形で置いています**。使う場所を確かめてから実行してください。
- 抜粋である以上、上から順に実行するだけで通るとは限りません。データの取得先やパスは、お手元の環境に合わせてください。
- **教育・研究のためのコードです。患者データをこのノートブックに置かないでください。**

リポジトリ: https://github.com/kewel-corp/book-basic

## 21.1　元画像・正解・予測を並べる

In [ ]:
import matplotlib.pyplot as plt

def show_triplet(image, gt_mask, pred_mask, save=None):
    fig, ax = plt.subplots(1, 3, figsize=(15, 5))
    ax[0].imshow(image, cmap="gray");            ax[0].set_title("input")
    ax[1].imshow(image, cmap="gray")
    ax[1].imshow(gt_mask, cmap="Reds", alpha=0.4); ax[1].set_title("ground truth")
    ax[2].imshow(image, cmap="gray")
    ax[2].imshow(pred_mask, cmap="Reds", alpha=0.4); ax[2].set_title("prediction")
    for a in ax: a.axis("off")
    if save: plt.savefig(save, dpi=150, bbox_inches="tight")
    plt.show()

## 21.2　オーバーレイ ― 元画像に重ねる

In [ ]:
import numpy as np
def overlay_mask(image, mask, color=(1, 0, 0), alpha=0.4):
    img = np.stack([image]*3, axis=-1) if image.ndim == 2 else image.copy()
    lo, hi = float(img.min()), float(img.max())
    img = (img - lo) / (hi - lo + 1e-8)          # 最小最大で[0,1]へ（CTのHUは負値を含む）
    colored = np.zeros_like(img); colored[mask > 0] = color
    return np.clip(img*(1-alpha) + colored*alpha, 0, 1)

## 21.3　3次元を、代表断面で見る

In [ ]:
def show_3d(volume, mask, spacing):
    """volumeとmaskは同じ座標系の(z, y, x)配列。
    spacingは、前処理後の配列に対応する(dz, dy, dx)をmm単位で渡す。
    解剖学的な軸方向は呼び出し側で確認しておく。
    """
    spacing = np.asarray(spacing, dtype=float)
    if spacing.shape != (3,) or not np.isfinite(spacing).all() or np.any(spacing <= 0):
        raise ValueError("spacingには有限で正の(dz, dy, dx)が必要")
    dz, dy, dx = spacing
    z = np.argmax(mask.sum(axis=(1, 2)))
    y = np.argmax(mask.sum(axis=(0, 2)))
    x = np.argmax(mask.sum(axis=(0, 1)))
    fig, ax = plt.subplots(1, 3, figsize=(15, 5))
    ax[0].imshow(overlay_mask(volume[z], mask[z]), aspect=dy/dx)
    ax[0].set_title("axial")
    ax[1].imshow(overlay_mask(volume[:, y], mask[:, y]), aspect=dz/dx)
    ax[1].set_title("coronal")
    ax[2].imshow(overlay_mask(volume[:, :, x], mask[:, :, x]), aspect=dz/dy)
    ax[2].set_title("sagittal")
    for a in ax:
        a.axis("off")
    plt.show()

## ミニプロジェクト ― 推論結果をオーバーレイして一括でPNG保存する

In [ ]:
from pathlib import Path
import numpy as np, torch
import matplotlib.pyplot as plt

@torch.no_grad()                                    # 推論なので勾配は不要（下の解説を参照）
def save_overlays(model, dataset, out_dir, spacing_mm, thr=0.5, device=None):
    """全画像を同じ正方画素の間隔へ統一済みの場合の例。
    spacing_mmは、リサイズ・リサンプリング後の画像で確認した間隔(mm)。
    間隔が不明、または症例ごと・行列方向で異なる場合は、この共通値の例を使わない。
    """
    import math, numbers
    if not (isinstance(spacing_mm, numbers.Real) and not isinstance(spacing_mm, bool)
            and math.isfinite(spacing_mm) and spacing_mm > 0):
        raise ValueError("確認済みの有限で正の画素間隔が必要です")
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")  # GPU未選択でも動く
    out = Path(out_dir); out.mkdir(parents=True, exist_ok=True)
    model.eval().to(device)                         # 評価モードを忘れない
    saved = 0
    for i in range(len(dataset)):
        image, case_id = dataset[i]                 # image: (1,H,W) の前処理済みテンソル
        prob = model(image[None].to(device)).softmax(1)[0, 1]  # 病変クラスの確率マップ
        pred = (prob > thr).cpu().numpy()           # 閾値でマスク化
        img  = image[0].cpu().numpy()               # 表示用に2次元へ

        area_mm2 = float(pred.sum()) * (spacing_mm ** 2)   # 予測面積を実寸で
        fig, ax = plt.subplots(figsize=(5, 5))
        ax.imshow(img, cmap="gray")
        ax.imshow(np.ma.masked_where(~pred, pred),  # 陰性画素は透明にして重ねる
                  cmap="autumn", alpha=0.45)
        ax.set_title(f"{case_id}  area={area_mm2:.0f} mm2")   # 図中は英語（日本語は□に化ける）
        ax.axis("off")
        fig.savefig(out / f"{case_id}_overlay.png", dpi=150, bbox_inches="tight")
        plt.close(fig)                              # 大量出力ではclose必須（メモリ解放）
        saved += 1
    print(f"{saved} 件を {out} に保存しました")

# 使い方：save_overlays(model, val_dataset, "review_out/", spacing_mm=0.7)